<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/process/closed_loop_process_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Closed Loop Process Optimization

Sweep a process setpoint with NeqSim and identify the lowest compressor power case that still meets an outlet-pressure target.


## Setup

Install imports and define a reusable gas fluid.


In [ ]:
# Install NeqSim when running in a fresh Colab session.
try:
    import neqsim
except ImportError:
    %pip install neqsim

import json
import pandas as pd
import matplotlib.pyplot as plt
from neqsim import jneqsim
from neqsim.thermo import TPflash

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
Stream = jneqsim.process.equipment.stream.Stream
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

def make_gas(temperature_c=25.0, pressure_bara=60.0):
    fluid = SystemSrkEos(273.15 + temperature_c, pressure_bara)
    fluid.addComponent("nitrogen", 0.01)
    fluid.addComponent("CO2", 0.02)
    fluid.addComponent("methane", 0.86)
    fluid.addComponent("ethane", 0.07)
    fluid.addComponent("propane", 0.03)
    fluid.addComponent("n-butane", 0.01)
    fluid.setMixingRule("classic")
    TPflash(fluid)
    fluid.initProperties()
    return fluid


## Build the Base Flowsheet

A compressor train with an inlet cooler is enough to demonstrate a closed-loop study.


In [ ]:
Heater = jneqsim.process.equipment.heatexchanger.Heater
Compressor = jneqsim.process.equipment.compressor.Compressor

fluid = make_gas(30.0, 55.0)
feed = Stream("Feed gas", fluid)
feed.setFlowRate(7500.0, "kg/hr")
cooler = Heater("Inlet cooler", feed)
compressor = Compressor("Compressor", cooler.getOutStream())

process = ProcessSystem()
for unit in [feed, cooler, compressor]:
    process.add(unit)


## Run a Setpoint Sweep

The sweep changes cooler outlet temperature and compressor outlet pressure, then records power.


In [ ]:
rows = []
for outlet_pressure in [80.0, 90.0, 100.0, 110.0, 120.0]:
    for cooler_temperature_c in [10.0, 20.0, 30.0]:
        cooler.setOutTemperature(273.15 + cooler_temperature_c)
        compressor.setOutletPressure(outlet_pressure)
        process.run()
        rows.append({
            "outlet_pressure_bara": outlet_pressure,
            "cooler_temperature_C": cooler_temperature_c,
            "power_kW": compressor.getPower("kW"),
        })
results = pd.DataFrame(rows)
results.sort_values("power_kW").head()


## Plot the Objective

Lower inlet temperature usually reduces compression power for the same pressure ratio.


In [ ]:
pivot = results.pivot(index="outlet_pressure_bara", columns="cooler_temperature_C", values="power_kW")
ax = pivot.plot(marker="o")
ax.set_xlabel("Compressor outlet pressure [bara]")
ax.set_ylabel("Compressor power [kW]")
ax.grid(True)
plt.show()
